## E1.1 Clasificar campos como Fact o Dimension

In [0]:
%sql

SELECT COUNT(*) as count FROM bootcamp.silver.propiedades;

SELECT
    moneda
FROM bootcamp.silver.propiedades
GROUP BY moneda;

In [0]:
%sql


    SELECT
        COUNT(*) AS registros,    
        COUNT(DISTINCT p.partido) AS cardinalidad_partido,
        COUNT(DISTINCT p.tipo_operacion) AS cardinalidad_tipo_operacion,
        COUNT(DISTINCT p.estado) AS cardinalidad_estado,    
        COUNT(DISTINCT p.moneda) AS cardinalidad_moneda,
        COUNT(DISTINCT p.precio) AS cardinalidad_precio,
        COUNT(DISTINCT p.metros_cuadrados_totales) AS cardinalidad_m2_totales,
        COUNT(DISTINCT p.url) AS cardinalidad_url
    FROM bootcamp.silver.propiedades p





## E1.5 - SCD Type 1 — Overwrite con MERGE

In [0]:
%sql
SELECT COUNT(*) as prev_count FROM bootcamp.gold.dim_zona;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW staging AS SELECT * FROM bootcamp.gold.dim_zona;

    

In [0]:
%sql

SELECT * FROM staging;

In [0]:
%sql

SELECT region FROM staging GROUP BY region;

In [0]:
%sql

MERGE INTO staging AS target
USING bootcamp.gold.dim_zona AS source
ON target.zona_id = source.zona_id AND source.region ="gba zona norte"
WHEN MATCHED THEN UPDATE SET 
    target.ciudad="Gran Buenos Aires";


In [0]:
%sql

SELECT * FROM staging;

In [0]:
%sql

SELECT ciudad FROM staging GROUP BY ciudad;

In [0]:
%sql

MERGE INTO bootcamp.gold.dim_zona as t
USING staging as s
ON t.zona_id=s.zona_id
WHEN MATCHED THEN UPDATE SET
    t.ciudad=s.ciudad;

In [0]:
%sql

SELECT COUNT(*) as count FROM bootcamp.gold.dim_zona;

## E1.6 - SCD Type 2 — Crear tabla con columnas de historización

In [0]:
%sql

DROP TABLE IF EXISTS bootcamp.gold.dim_zona_scd2;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_zona_scd2
(
    zona_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT "PK",    
    partido STRING NOT NULL COMMENT "Partido donde se encuentra la propiedad",
    region STRING NOT NULL COMMENT "Region donde se encuentra la propiedad",
    ciudad STRING NOT NULL COMMENT "Ciudad donde se encuentra la propiedad",
    provincia STRING NOT NULL COMMENT "Provincia donde se encuentra la propiedad" DEFAULT "Buenos Aires",
    pais string NOT NULL COMMENT "Pais donde se encuentra la propiedad" DEFAULT "Argentina",
    valid_from TIMESTAMP NOT NULL COMMENT "Fecha desde la cual es valida la informacion",
    valid_to TIMESTAMP NOT NULL DEFAULT "9999-12-31" COMMENT "Fecha hasta la cual es valida la informacion",
    is_current BOOLEAN NOT NULL DEFAULT true COMMENT "Indica si la informacion es actual o no",    
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY(zona_id)
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Zona SCD Type 2";


In [0]:
%sql

INSERT OVERWRITE bootcamp.gold.dim_zona_scd2 (    
    partido,
    region,
    ciudad,
    provincia,
    pais,
    valid_from
)
SELECT 
        DISTINCT 
        partido, 
        region,
        CASE
            WHEN region ='capital federal' THEN 'CABA'
            ELSE 'GBA'
        END as ciudad,
        CASE
            WHEN region ='capital federal' THEN 'CABA'
            ELSE 'Buenos Aires'
        END as provincia,
        'Argentina' as pais,
        CURRENT_TIMESTAMP() as valid_from
       
FROM bootcamp.silver.propiedades
WHERE partido IS NOT NULL AND region IS NOT NULL
ORDER BY partido, region;
    

In [0]:
%sql

SELECT
    COUNT(*) as reg_is_current_false
FROM bootcamp.gold.dim_zona_scd2
WHERE is_current = false;

## E1.7 - SCD Type 2 — Historizar un cambio (manual)

In [0]:
%sql
UPDATE bootcamp.gold.dim_zona_scd2
    SET valid_to = CURRENT_TIMESTAMP(),
    is_current = FALSE
WHERE partido = 'capital federal'
AND is_current = TRUE;

In [0]:
%sql
INSERT INTO bootcamp.gold.dim_zona_scd2
(partido, region, ciudad,
valid_from)
VALUES (
'capital federal',
'capital federal',
'CABA',
CURRENT_TIMESTAMP()
);


In [0]:
%sql

SELECT * FROM bootcamp.gold.dim_zona_scd2 WHERE partido='capital federal' ORDER BY valid_from;

## E1.8 - SCD Type 2 — Historizar con MERGE (masivo)

In [0]:
%sql
SELECT DISTINCT partido FROM bootcamp.silver.propiedades ORDER BY partido;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW staging_zona_scd2 AS
SELECT * FROM VALUES
("avellaneda","gba zona sur", "CABA Sur"),
("barrio-nuevo","capital federal","CABA")
AS t(partido, region, ciudad);

In [0]:
SELECT * from staging_zona_scd2;

In [0]:
SELECT
    dest.*,
    src.*
FROM bootcamp.gold.dim_zona_scd2 dest
LEFT JOIN staging_zona_scd2 src
ON dest.partido==src.partido AND dest.region==src.region AND dest.is_current==true;

In [0]:
%sql

MERGE INTO bootcamp.gold.dim_zona_scd2 dest
USING staging_zona_scd2 src
ON dest.partido=src.partido AND dest.region=src.region AND dest.is_current=true
WHEN MATCHED AND dest.ciudad <> src.ciudad THEN 
UPDATE SET valid_to=CURRENT_TIMESTAMP(), is_current=FALSE
WHEN NOT MATCHED THEN
INSERT (partido, region, ciudad, valid_from)
VALUES (src.partido, src.region, src.ciudad, CURRENT_TIMESTAMP());

In [0]:
SELECT 
    1,
    src.*
FROM bootcamp.gold.dim_zona_scd2 dest
LEFT JOIN staging_zona_scd2 src
WHERE dest.partido==src.partido AND dest.region==src.region AND dest.is_current==true;

-- SELECT 1 FROM staging_zona_scd2;
-- SELECT 1 
--     FROM bootcamp.gold.dim_zona_scd2  s
--     WHERE partido = s.partido 
--       AND region = s.region 
--       AND is_current = false;

In [0]:
INSERT INTO bootcamp.gold.dim_zona_scd2
(partido, region, ciudad, valid_from)
SELECT     
    src.partido,
    src.region,
    src.ciudad,    
    CURRENT_TIMESTAMP() AS valid_from
FROM staging_zona_scd2 src
WHERE NOT EXISTS (
    SELECT 1 
    FROM bootcamp.gold.dim_zona_scd2 z
    WHERE z.partido == src.partido 
      AND z.region == src.region 
      AND z.is_current == true
);

In [0]:
%sql
select * from bootcamp.gold.dim_zona_scd2 where partido like 'avellaneda' OR partido like 'barrio-nuevo';

## E1.9 SCD Type 3 - Guardar valor anterior con MERGE

In [0]:
DROP TABLE IF EXISTS bootcamp.gold.dim_zona_scd3;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_zona_scd3 (
  zona_id BIGINT GENERATED ALWAYS AS IDENTITY (INCREMENT BY 1 START WITH 1),
  partido string NOT NULL,
  region string NOT NULL,
  ciudad string NOT NULL,
  provincia string NOT NULL,
  pais string NOT NULL,
  ciudad_anterior string NOT NULL,
  fecha_ultima_actualizacion TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
  _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
  PRIMARY KEY(zona_id)
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Zona SCD Type 3";

In [0]:
%sql
INSERT OVERWRITE bootcamp.gold.dim_zona_scd3 (    
    partido,
    region,
    ciudad,
    provincia,
    pais,
    ciudad_anterior
)
SELECT 
        DISTINCT 
        partido, 
        region,
        CASE
            WHEN region ='capital federal' THEN 'CABA'
            ELSE 'GBA'
        END as ciudad,
        CASE
            WHEN region ='capital federal' THEN 'CABA'
            ELSE 'Buenos Aires'
        END as provincia,
        'Argentina' as pais,        
        "" as ciudad_anterior
FROM bootcamp.silver.propiedades
WHERE partido IS NOT NULL AND region IS NOT NULL
ORDER BY partido, region;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW staging_zona_scd3 AS
SELECT * FROM VALUES
("avellaneda","gba zona sur", "CABA Sur"),
("barrio-nuevo","capital federal","CABA")
AS t(partido, region, ciudad);

In [0]:
select * from staging_zona_scd3;

In [0]:
merge into bootcamp.gold.dim_zona_scd3 as dest
using staging_zona_scd3 as src
on dest.partido = src.partido and dest.region = src.region
when matched then update set 
dest.ciudad_anterior = dest.ciudad,
dest.ciudad=src.ciudad,
dest.fecha_ultima_actualizacion=CURRENT_TIMESTAMP()
when not matched then insert 
(partido, region, ciudad, provincia, pais, ciudad_anterior)
values (
    src.partido, 
    src.region, 
    src.ciudad, 
    CASE
        WHEN region ='capital federal' THEN 'CABA'
        ELSE 'Buenos Aires'
    END,
    'Argentina', 
    ""
    );


In [0]:
%sql

select * from bootcamp.gold.dim_zona_scd3 where ciudad_anterior <> '';

## E2.2 dim_tipo_operacion + dim_caracteristicas — DDL + ETL

In [0]:
DROP TABLE IF EXISTS bootcamp.gold.dim_tipo_operacion;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_tipo_operacion(
    tipo_operacion_id BIGINT GENERATED ALWAYS AS IDENTITY (INCREMENT BY 1 START WITH 1),
    tipo_operacion string NOT NULL,   
    categoria string NOT NULL,
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Tipo Operacion SCD1";

In [0]:
DROP TABLE IF EXISTS bootcamp.gold.dim_caracteristicas;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_caracteristicas(
    caracteristicas_id BIGINT GENERATED ALWAYS AS IDENTITY (INCREMENT BY 1 START WITH 1),
    estado string NOT NULL,
    cochera boolean NOT NULL,
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Caracteristicas SCD1";

In [0]:
%sql

-- select distinct tipo_operacion from bootcamp.silver.propiedades;

-- select distinct moneda from bootcamp.silver.propiedades;

-- select distinct precio from bootcamp.silver.propiedades;